# GraphRAG Query Notebook (Extended: Vector + Graph + Neighboring Chunks)

Hybrid retrieval over the Neo4j knowledge graph built by
`chunk_embed_graph_pipeline.ipynb`:

1. **Vector leg** — semantic search over `(:Chunk).text_embedding` (bge-m3, cosine).
2. **Graph leg** — match entity names in the question, expand through KG
   relationships, collect `MENTIONED_IN` chunks.
3. **Neighbor expansion** — follow `previous_chunk_id` / `next_chunk_id`
   chain links to include adjacent document chunks (default 1 hop).
4. **Generation** — feed the fused, document-ordered context to `rnj-1:latest`.

Usage: run all cells, then call `ask("your question")` in the last cell.
Adjust `top_k`, `kg_hops`, and `neighbor_hops` as needed.

# 1. Configuration

In [ ]:
# Neo4j connection
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "neo4jadmin"
NEO4J_DATABASE = "neo4j"

# Models
EMBED_MODEL   = "BAAI/bge-m3"
EXTRACT_MODEL = "rnj-1:latest"

# Ontology labels (for KG expansion scoping)
KG_LABELS = [
    "Topic", "Subtopic", "Concept", "Definition",
    "Formula", "Theorem", "Example",
]

# 2. Setup (driver, embeddings, document chain order)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"Driver ready ({NEO4J_URI})  |  Embeddings loaded ({EMBED_MODEL})")

In [ ]:
def get_chunk_order():
    """Walk the previous_chunk_id / next_chunk_id chain to produce an
    ordered list of chunk_ids and a position lookup dict."""
    with driver.session(database=NEO4J_DATABASE) as s:
        rows = s.run(
            "MATCH (c:Chunk) "
            "RETURN c.chunk_id AS cid, "
            "       c.previous_chunk_id AS prev, "
            "       c.next_chunk_id AS next"
        ).data()
    by_id = {r["cid"]: (r["prev"], r["next"]) for r in rows}
    start = next(cid for cid, (prev, _) in by_id.items() if prev is None)
    order = [start]
    while by_id.get(order[-1], (None, None))[1]:
        order.append(by_id[order[-1]][1])
    return order, {cid: i for i, cid in enumerate(order)}

CHAIN_ORDER, CHAIN_POS = get_chunk_order()
print(f"Document chain built: {len(CHAIN_ORDER)} chunks, "
      f"first={CHAIN_ORDER[0][:12]}… last={CHAIN_ORDER[-1][:12]}…")

# 3. Retrieval primitives

In [ ]:
def vector_search(question, top_k=5):
    """Embed question, search chunk_embeddings vector index."""
    qv = embeddings.embed_query(question)
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(
            "CALL db.index.vector.queryNodes('chunk_embeddings', $k, $qv) "
            "YIELD node, score "
            "RETURN node.chunk_id AS id, node.text AS text, "
            "       node.header AS header, score",
            k=top_k, qv=qv,
        ).data()

In [ ]:
def match_entity_names(question):
    """Return entity ids whose name matches terms in the question."""
    with driver.session(database=NEO4J_DATABASE) as s:
        entities = s.run(
            "MATCH (n) WHERE n.name IS NOT NULL "
            "RETURN n.id AS id, n.name AS name"
        ).data()
    q = question.lower()
    tokens = {t for t in q.split() if len(t) > 2 and t.isalnum()}
    matched = []
    for row in entities:
        low = row["name"].lower()
        if low in q:
            matched.append(row["id"])
            continue
        name_tokens = {t for t in low.split() if len(t) > 2 and t.isalnum()}
        if name_tokens and name_tokens.issubset(tokens):
            matched.append(row["id"])
    return matched

In [ ]:
def expand_entities(entity_ids, hops=1):
    """Expand entity ids through KG relationships (scoped to KG labels)."""
    if hops <= 0 or not entity_ids:
        return set(entity_ids)
    label_pred = " OR ".join(f"x:{l}" for l in KG_LABELS)
    with driver.session(database=NEO4J_DATABASE) as s:
        rows = s.run(
            "MATCH (e) WHERE e.id IN $ids "
            "WITH collect(DISTINCT e) AS ents "
            "UNWIND ents AS e "
            f"MATCH (e)-[*1..{int(hops)}]-(x) "
            f"WHERE {label_pred} "
            "RETURN DISTINCT x.id AS id",
            ids=entity_ids,
        ).data()
    return set(entity_ids) | {r["id"] for r in rows}

In [ ]:
def get_mentioned_chunks(entity_ids):
    """Collect all MENTIONED_IN chunks from a set of entity ids."""
    if not entity_ids:
        return set()
    with driver.session(database=NEO4J_DATABASE) as s:
        rows = s.run(
            "MATCH (e) WHERE e.id IN $ids "
            "MATCH (e)-[:MENTIONED_IN]->(c:Chunk) "
            "RETURN DISTINCT c.chunk_id AS cid",
            ids=list(entity_ids),
        ).data()
    return {r["cid"] for r in rows}

In [ ]:
def expand_neighbors(chunk_ids, hops=1):
    """Follow previous_chunk_id / next_chunk_id chain links from initial
    chunk_ids. Returns the set of newly added neighbor chunk ids."""
    all_ids = set(chunk_ids)
    added_total = set()
    for hop in range(hops):
        current = list(all_ids)
        if not current:
            break
        with driver.session(database=NEO4J_DATABASE) as s:
            rows = s.run(
                "MATCH (c:Chunk) WHERE c.chunk_id IN $ids "
                "RETURN c.previous_chunk_id AS prev, c.next_chunk_id AS next",
                ids=current,
            ).data()
        new_ids = set()
        for r in rows:
            if r["prev"]:
                new_ids.add(r["prev"])
            if r["next"]:
                new_ids.add(r["next"])
        added = new_ids - all_ids
        all_ids |= new_ids
        added_total |= added
        if added:
            print(f"  hop {hop + 1}: +{len(added)} neighbor chunk(s)")
    return added_total

# 4. Hybrid retrieval (vector + graph + neighbors)

In [ ]:
def retrieve_hybrid(question, top_k=5, kg_hops=1, neighbor_hops=1):
    """Three-leg retrieval fused and ordered by document position."""
    # --- 1. vector search ---
    v_chunks = vector_search(question, top_k)
    v_ids = {c["id"] for c in v_chunks}

    # --- 2. graph entity match + KG expansion ---
    entity_ids = match_entity_names(question)
    g_ids = set()
    if entity_ids:
        expanded = expand_entities(entity_ids, kg_hops)
        g_ids = get_mentioned_chunks(expanded)

    # --- 3. neighborhood expansion on all retrieved chunks ---
    initial_ids = v_ids | g_ids
    neighbor_ids = expand_neighbors(initial_ids, neighbor_hops)

    # --- 4. fetch full chunk data, dedup, order by document position ---
    all_ids = initial_ids | neighbor_ids
    with driver.session(database=NEO4J_DATABASE) as s:
        chunks = s.run(
            "MATCH (c:Chunk) WHERE c.chunk_id IN $ids "
            "RETURN c.chunk_id AS id, c.text AS text, c.header AS header",
            ids=list(all_ids),
        ).data()
    chunks.sort(key=lambda r: CHAIN_POS.get(r["id"], 999))

    # source metadata (vector / graph / neighbor)
    sources = {}
    for c in v_chunks:
        sources[c["id"]] = ("vector", c["score"])
    for cid in g_ids:
        sources.setdefault(cid, ("graph", None))
    for cid in neighbor_ids:
        sources.setdefault(cid, ("neighbor", None))

    return chunks, sources

# 5. LLM generation

In [ ]:
def generate_answer(question, chunks, model=EXTRACT_MODEL, num_ctx=8192):
    """Feed document-ordered chunks as context to the LLM."""
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import ChatPromptTemplate

    llm = ChatOllama(model=model, temperature=0, num_ctx=num_ctx)

    if chunks:
        context = "\n\n".join(
            f"[Chunk {i + 1}] (header: {c['header'] or 'n/a'})\n{c['text']}"
            for i, c in enumerate(chunks)
        )
        system = (
            "You are a physics educator. Answer the student's question using ONLY "
            "the provided source chunks. If the answer cannot be found, say so. "
            "Cite the chunk numbers you used, e.g. (chunk 1, chunk 4)."
        )
    else:
        context = "No relevant chunks were retrieved."
        system = (
            "You are a physics educator. No relevant context was retrieved; "
            "state that you cannot answer from the knowledge base."
        )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", "Context:\n{context}\n\nQuestion:\n{question}"),
    ])
    return llm.invoke(prompt.format(context=context, question=question)).content

# 6. Ask function + run

In [ ]:
def ask(question, top_k=5, kg_hops=1, neighbor_hops=1):
    """Full pipeline: retrieve → show breakdown → generate answer."""
    print(f"Question: {question}\n")
    chunks, sources = retrieve_hybrid(question, top_k, kg_hops, neighbor_hops)

    print(f"\nRetrieved {len(chunks)} chunks total:\n")
    for i, c in enumerate(chunks):
        src, score = sources.get(c["id"], ("?", None))
        tag = f"score={score:.3f}" if score is not None else src
        print(f"  [{src:<8} {tag:>14}]  chunk {i + 1}: {c['id'][:12]}…  header={c['header']!r}")

    print()
    answer = generate_answer(question, chunks)
    print("=== ANSWER ===")
    print(answer)

In [ ]:
# --- try it ---
ask("What is gravitational potential energy?")